# WeatherPre · Aurora on Colab (GPU runtime)
Same code path as the HF Space tab (`weatherpre.adapters.aurora_run`): upstream `microsoft-aurora` + real initial conditions. Weights (MIT) are downloaded from `microsoft/aurora` at run time.

Status: the identical module ran on the HF ZeroGPU Space (see repo README); this notebook itself has **not** been executed on Colab.

In [ ]:
!pip -q install microsoft-aurora ecmwf-opendata cfgrib eccodes gcsfs zarr xarray netCDF4 huggingface_hub
!git clone -q --depth 1 https://github.com/GISWLH/WeatherPre && cd WeatherPre && echo cloned

In [ ]:
import sys; sys.path.insert(0, 'WeatherPre')
# historic: ERA5 initial conditions from WeatherBench 2 (aurora-0.25-pretrained)
!cd WeatherPre && python -m weatherpre.adapters.aurora_run --init 2020-10-01T00 --steps 8 --source era5 --variant pretrained --out /content/aurora_2020-10-01.nc

In [ ]:
# latest: IFS HRES analyses from ECMWF open data (last ~4 days), HRES-T0 fine-tuned checkpoint
import datetime as dt
from ecmwf.opendata import Client
init = Client(source='ecmwf', model='ifs').latest(type='fc', stream='oper', step=0, param='2t', levtype='sfc')
print('latest IFS cycle', init)
!cd WeatherPre && python -m weatherpre.adapters.aurora_run --init {init:%Y-%m-%dT%H} --steps 8 --source ifs --variant finetuned --out /content/aurora_latest.nc

In [ ]:
import xarray as xr, matplotlib.pyplot as plt
ds = xr.open_dataset('/content/aurora_latest.nc')
ds.geopotential.sel(level=500).isel(time=0, prediction_timedelta=-1).plot()
plt.title('Aurora z500, last lead'); plt.show()